# DGGR Notebooks

This notebook is part of the **Deep Generative Genre Remastering (DGGR)** project.

## How To Run
- Prefer running from the repo root so relative paths resolve.
- Most notebooks assume you have the Lab artifacts under `saves/` and `saves2/` (ignored by git).
- See `docs/` for setup, data layout, and reproduction notes.

## Notes
- Outputs are intentionally stripped for version control cleanliness.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path


def _find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for _ in range(8):
        if (p / "dggr").exists():
            return p
        p = p.parent
    return (start or Path.cwd()).resolve()


REPO_ROOT = _find_repo_root()
DATA_ROOT = Path(os.environ.get("DGGR_DATA_ROOT", str(REPO_ROOT / "data")))
MANIFESTS_ROOT = Path(os.environ.get("DGGR_MANIFESTS_ROOT", str(DATA_ROOT / "_lab1_manifests")))

print("REPO_ROOT:", REPO_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("MANIFESTS_ROOT:", MANIFESTS_ROOT)


# Lab 4 Longform Coherence Workbench

This notebook is a full control surface for:
1. Diffusion training commands (V2 and V3)
2. Coherence-constrained longform synthesis
3. Full-song style transfer test
4. Cohesion diagnostics and listening

Core inference backend: `run_lab4_longform_coherence.py`


## 1) Environment Setup

In [ ]:
from __future__ import annotations

import json
import shlex
import subprocess
import sys
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
import IPython.display as ipd

ROOT = Path(r"Z:/328/CMPUT328-A2/codexworks/301/414-pl1")
LAB3 = ROOT / "lab 3"
LAB4 = ROOT / "lab 4"
assert LAB3.exists(), f"Missing lab3 dir: {LAB3}"
assert LAB4.exists(), f"Missing lab4 dir: {LAB4}"

print("Python:", sys.executable)
print("Lab3:", LAB3)
print("Lab4:", LAB4)


In [ ]:
def run_cmd(cmd: list[str], cwd: Path | None = None, check: bool = True):
    print("\n$", " ".join(shlex.quote(str(x)) for x in cmd))
    proc = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd is not None else None,
        check=False,
        text=True,
    )
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed with code {proc.returncode}")
    return proc.returncode


## 2) Shared Paths

Update these only if your checkpoints move.

In [ ]:
PATHS = {
    "cache_dir": ROOT / "saves2/lab3_diffusion/run_d001/cache",
    "lab1_checkpoint": ROOT / "saves/lab1_run_combo_af_gate_exit_v2/latest.pt",
    "diffusion_v2_ckpt": ROOT / "saves2/lab3_diffusion/run_d002/checkpoints/epoch_006.pt",
    "diffusion_v3_ckpt": ROOT / "saves2/lab3_diffusion/run_d003/checkpoints/latest.pt",
    "song_path": Path(r"C:/Users/Ahmed/Downloads/Milky & Mall Grab - Just The Way You Are.flac"),
    "coherence_script": LAB4 / "run_lab4_longform_coherence.py",
    "train_v2_script": LAB3 / "run_lab3_diffusion_v2.py",
    "train_v3_script": LAB3 / "run_lab3_diffusion_v3.py",
}

for k, v in PATHS.items():
    print(f"{k}: {v}  | exists={Path(v).exists()}")


## 3) Training Knobs (V2 + V3)

These are command-level knobs. Set `RUN_TRAIN_V2` or `RUN_TRAIN_V3` to `True` to actually launch training from notebook.

In [ ]:
TRAIN_V2 = {
    "out_dir": ROOT / "saves2/lab3_diffusion/run_d002",
    "epochs": 60,
    "lr": 2e-4,
    "batch_size": 4,
    "grad_accum": 4,
    "max_frames": 256,
    "ema_decay": 0.9999,
    "cfg_dropout_p": 0.10,
    "ddim_steps": 50,
    "guidance_scale": 2.0,
    "device": "auto",
}

TRAIN_V3 = {
    "out_dir": ROOT / "saves2/lab3_diffusion/run_d003",
    "v2_checkpoint": ROOT / "saves2/lab3_diffusion/run_d002/checkpoints/epoch_006.pt",
    "epochs": 20,
    "lr": 1e-4,
    "disc_lr": 2e-4,
    "batch_size": 4,
    "grad_accum": 4,
    "max_frames": 256,
    "ema_decay": 0.999,
    "cfg_dropout_p": 0.15,
    "disc_warmup_steps": 500,
    "adv_weight": 0.1,
    "fm_weight": 0.5,
    "ddim_steps": 50,
    "guidance_scale": 2.0,
    "device": "auto",
}

RUN_TRAIN_V2 = False
RUN_TRAIN_V3 = False


In [ ]:
def build_train_v2_cmd(cfg: dict) -> list[str]:
    return [
        sys.executable,
        str(PATHS["train_v2_script"]),
        "--cache-dir", str(PATHS["cache_dir"]),
        "--out-dir", str(cfg["out_dir"]),
        "--epochs", str(cfg["epochs"]),
        "--lr", str(cfg["lr"]),
        "--batch-size", str(cfg["batch_size"]),
        "--grad-accum", str(cfg["grad_accum"]),
        "--max-frames", str(cfg["max_frames"]),
        "--ema-decay", str(cfg["ema_decay"]),
        "--cfg-dropout-p", str(cfg["cfg_dropout_p"]),
        "--ddim-steps", str(cfg["ddim_steps"]),
        "--guidance-scale", str(cfg["guidance_scale"]),
        "--device", str(cfg["device"]),
    ]


def build_train_v3_cmd(cfg: dict) -> list[str]:
    return [
        sys.executable,
        str(PATHS["train_v3_script"]),
        "--cache-dir", str(PATHS["cache_dir"]),
        "--out-dir", str(cfg["out_dir"]),
        "--v2-checkpoint", str(cfg["v2_checkpoint"]),
        "--epochs", str(cfg["epochs"]),
        "--lr", str(cfg["lr"]),
        "--disc-lr", str(cfg["disc_lr"]),
        "--batch-size", str(cfg["batch_size"]),
        "--grad-accum", str(cfg["grad_accum"]),
        "--max-frames", str(cfg["max_frames"]),
        "--ema-decay", str(cfg["ema_decay"]),
        "--cfg-dropout-p", str(cfg["cfg_dropout_p"]),
        "--disc-warmup-steps", str(cfg["disc_warmup_steps"]),
        "--adv-weight", str(cfg["adv_weight"]),
        "--fm-weight", str(cfg["fm_weight"]),
        "--ddim-steps", str(cfg["ddim_steps"]),
        "--guidance-scale", str(cfg["guidance_scale"]),
        "--device", str(cfg["device"]),
    ]


cmd_v2 = build_train_v2_cmd(TRAIN_V2)
cmd_v3 = build_train_v3_cmd(TRAIN_V3)

print("V2 command:", " ".join(shlex.quote(x) for x in cmd_v2))
print("V3 command:", " ".join(shlex.quote(x) for x in cmd_v3))


In [ ]:
if RUN_TRAIN_V2:
    run_cmd(build_train_v2_cmd(TRAIN_V2), cwd=LAB3, check=True)
if RUN_TRAIN_V3:
    run_cmd(build_train_v3_cmd(TRAIN_V3), cwd=LAB3, check=True)

if not RUN_TRAIN_V2 and not RUN_TRAIN_V3:
    print("Training not launched. Set RUN_TRAIN_V2/RUN_TRAIN_V3=True to run.")


## 4) Synthesis Knobs (Coherence-First)

This is the main Lab 4 inference configuration. Defaults are set for a full-song run.


In [ ]:
SYNTH = {
    "checkpoint": PATHS["diffusion_v2_ckpt"],
    "source_audio": PATHS["song_path"],
    "source_start_sec": 0.0,
    "source_seconds": 0.0,   # <=0 means auto full duration
    "source_genre": "hiphop_xtc",
    "target_genre": "baroque_classical",
    "chunk_seconds": 3.0,
    "overlap_seconds": 0.5,
    "n_frames": 256,
    "t_start": 350,
    "t_start_end": 280,
    "reanchor_every": 12,
    "reanchor_t_start": 220,
    "ddim_steps": 50,
    "guidance_scale": 1.8,
    "style_strength": 0.75,
    "prefix_blend": 1.0,
    "source_prefix_blend": 0.25,
    "source_mel_blend": 0.10,
    "hf_source_blend": 0.20,
    "hf_start_bin": 56,
    "mel_time_smooth": 5,
    "mel_freq_smooth": 0,
    "assemble_domain": "mel",  # mel recommended to reduce phase/static seams
    "eta": 0.0,
    "seed": 328,
    "device": "auto",
    "out_dir": ROOT / "saves2/lab4_longform_coherence/fullsong_stable",
}

print(json.dumps({k: str(v) if isinstance(v, Path) else v for k, v in SYNTH.items()}, indent=2))


In [ ]:
def build_synth_cmd(cfg: dict) -> list[str]:
    return [
        sys.executable,
        str(PATHS["coherence_script"]),
        "--cache-dir", str(PATHS["cache_dir"]),
        "--checkpoint", str(cfg["checkpoint"]),
        "--lab1-checkpoint", str(PATHS["lab1_checkpoint"]),
        "--source-audio", str(cfg["source_audio"]),
        "--source-start-sec", str(cfg["source_start_sec"]),
        "--source-seconds", str(cfg["source_seconds"]),
        "--source-genre", str(cfg["source_genre"]),
        "--target-genre", str(cfg["target_genre"]),
        "--chunk-seconds", str(cfg["chunk_seconds"]),
        "--overlap-seconds", str(cfg["overlap_seconds"]),
        "--n-frames", str(cfg["n_frames"]),
        "--t-start", str(cfg["t_start"]),
        "--t-start-end", str(cfg["t_start_end"]),
        "--reanchor-every", str(cfg["reanchor_every"]),
        "--reanchor-t-start", str(cfg["reanchor_t_start"]),
        "--ddim-steps", str(cfg["ddim_steps"]),
        "--guidance-scale", str(cfg["guidance_scale"]),
        "--style-strength", str(cfg["style_strength"]),
        "--prefix-blend", str(cfg["prefix_blend"]),
        "--source-prefix-blend", str(cfg["source_prefix_blend"]),
        "--source-mel-blend", str(cfg["source_mel_blend"]),
        "--hf-source-blend", str(cfg["hf_source_blend"]),
        "--hf-start-bin", str(cfg["hf_start_bin"]),
        "--mel-time-smooth", str(cfg["mel_time_smooth"]),
        "--mel-freq-smooth", str(cfg["mel_freq_smooth"]),
        "--assemble-domain", str(cfg["assemble_domain"]),
        "--eta", str(cfg["eta"]),
        "--seed", str(cfg["seed"]),
        "--device", str(cfg["device"]),
        "--out-dir", str(cfg["out_dir"]),
    ]

synth_cmd = build_synth_cmd(SYNTH)
print(" ".join(shlex.quote(x) for x in synth_cmd))


## 4b) De-Warble Presets

Set `PROFILE` then run this cell to update `SYNTH` before full-song synthesis.

In [ ]:
PROFILE = "stable"  # options: stable, aggressive_style, max_preserve

PRESETS = {
    "stable": {
        "guidance_scale": 1.8,
        "style_strength": 0.75,
        "t_start": 350,
        "t_start_end": 280,
        "reanchor_every": 12,
        "reanchor_t_start": 220,
        "source_prefix_blend": 0.25,
        "source_mel_blend": 0.10,
        "hf_source_blend": 0.20,
        "hf_start_bin": 56,
        "mel_time_smooth": 5,
        "mel_freq_smooth": 0,
        "assemble_domain": "mel",
    },
    "aggressive_style": {
        "guidance_scale": 2.1,
        "style_strength": 0.90,
        "t_start": 360,
        "t_start_end": 320,
        "reanchor_every": 14,
        "reanchor_t_start": 240,
        "source_prefix_blend": 0.20,
        "source_mel_blend": 0.06,
        "hf_source_blend": 0.10,
        "hf_start_bin": 60,
        "mel_time_smooth": 3,
        "mel_freq_smooth": 0,
        "assemble_domain": "mel",
    },
    "max_preserve": {
        "guidance_scale": 1.6,
        "style_strength": 0.60,
        "t_start": 320,
        "t_start_end": 250,
        "reanchor_every": 8,
        "reanchor_t_start": 200,
        "source_prefix_blend": 0.35,
        "source_mel_blend": 0.15,
        "hf_source_blend": 0.28,
        "hf_start_bin": 52,
        "mel_time_smooth": 7,
        "mel_freq_smooth": 3,
        "assemble_domain": "mel",
    },
}

assert PROFILE in PRESETS, f"Unknown PROFILE: {PROFILE}"
SYNTH.update(PRESETS[PROFILE])
print("Applied profile:", PROFILE)
print(json.dumps({k: str(v) if isinstance(v, Path) else v for k, v in SYNTH.items()}, indent=2))


## 5) Full-Song Synthesis Run

This cell executes full-song style transfer with coherence constraints.


In [ ]:
RUN_FULL_SONG = True
if RUN_FULL_SONG:
    run_cmd(build_synth_cmd(SYNTH), cwd=LAB4, check=True)
else:
    print("Set RUN_FULL_SONG=True to synthesize.")


## 6) Load Results + Metrics

In [ ]:
OUT = Path(SYNTH["out_dir"])
coh_json = OUT / "coherence_metrics.json"
coh_wav = OUT / "longform_coherent.wav"
src_wav = OUT / "source.wav"

print("Output dir:", OUT)
print("Source wav exists:", src_wav.exists())
print("Generated wav exists:", coh_wav.exists())
print("Metrics exists:", coh_json.exists())

if coh_json.exists():
    metrics = json.loads(coh_json.read_text(encoding="utf-8"))
    print(json.dumps(metrics, indent=2))
else:
    metrics = {}


In [ ]:
if src_wav.exists() and coh_wav.exists():
    y_src, sr_src = librosa.load(src_wav, sr=None, mono=True)
    y_gen, sr_gen = librosa.load(coh_wav, sr=None, mono=True)
    print(f"source: {len(y_src)/sr_src:.2f}s @ {sr_src}")
    print(f"gen:    {len(y_gen)/sr_gen:.2f}s @ {sr_gen}")

    # simple spectrogram compare
    m_src = librosa.power_to_db(librosa.feature.melspectrogram(y=y_src, sr=sr_src, n_mels=80, hop_length=256), ref=np.max)
    m_gen = librosa.power_to_db(librosa.feature.melspectrogram(y=y_gen, sr=sr_gen, n_mels=80, hop_length=256), ref=np.max)

    fig, ax = plt.subplots(2, 1, figsize=(14, 8), sharex=False)
    ax[0].imshow(m_src, origin="lower", aspect="auto", cmap="magma")
    ax[0].set_title("Source mel")
    ax[1].imshow(m_gen, origin="lower", aspect="auto", cmap="magma")
    ax[1].set_title("Generated mel (coherence constrained)")
    plt.tight_layout()
    plt.show()
else:
    print("Missing source/generated wavs.")


## 7) Listen

In [ ]:
if src_wav.exists():
    print("Source")
    display(ipd.Audio(str(src_wav)))
if coh_wav.exists():
    print("Generated")
    display(ipd.Audio(str(coh_wav)))


## 8) Optional Coherence Sweep

Quick ablation over continuity knobs. Set `RUN_SWEEP=True` to execute.


In [ ]:
SWEEP = {
    "t_start": [250, 350],
    "prefix_blend": [0.7, 1.0],
    "ddim_steps": [30, 50],
}
RUN_SWEEP = False


In [ ]:
if RUN_SWEEP:
    base = dict(SYNTH)
    for t in SWEEP["t_start"]:
        for pb in SWEEP["prefix_blend"]:
            for ds in SWEEP["ddim_steps"]:
                tag = f"t{t}_pb{pb}_ds{ds}".replace('.', 'p')
                cfg = dict(base)
                cfg["t_start"] = t
                cfg["prefix_blend"] = pb
                cfg["ddim_steps"] = ds
                cfg["out_dir"] = ROOT / f"saves2/lab4_longform_coherence/sweeps/{tag}"
                print("\nRunning", tag)
                run_cmd(build_synth_cmd(cfg), cwd=LAB4, check=True)
else:
    print("Sweep not run. Set RUN_SWEEP=True to execute.")


## 9) Genre Space + Target Vector Audit

This section shows exactly what style space the diffusion model can target, and where your selected target genre sits in that space.

In [ ]:
import json
import numpy as np
import pandas as pd

cache_dir = Path(PATHS["cache_dir"])
genre_map = json.loads((cache_dir / "diff_genre_to_idx.json").read_text(encoding="utf-8"))
idx_to_genre = {int(v): str(k) for k, v in genre_map.items()}

genre_idx = np.load(cache_dir / "diff_genre_idx.npy", mmap_mode="r")
z_style = np.load(cache_dir / "diff_z_style.npy", mmap_mode="r")

print("Learned diffusion genres:", len(genre_map))
print(sorted(genre_map.keys()))
print("Selected target genre:", SYNTH["target_genre"])

centroids = {}
for g, gi in genre_map.items():
    mask = genre_idx == int(gi)
    c = np.asarray(z_style[mask], dtype=np.float32).mean(axis=0)
    c = c / (np.linalg.norm(c) + 1e-8)
    centroids[g] = c

# Cosine similarity matrix between genre centroids
g_names = sorted(genre_map.keys())
M = np.zeros((len(g_names), len(g_names)), dtype=np.float32)
for i, a in enumerate(g_names):
    for j, b in enumerate(g_names):
        M[i, j] = float(np.dot(centroids[a], centroids[b]))

sim_df = pd.DataFrame(M, index=g_names, columns=g_names)
print("\nGenre centroid cosine similarity (lower off-diagonal = better separation):")
display(sim_df.style.background_gradient(cmap="viridis"))

if SYNTH["target_genre"] in centroids:
    tgt = centroids[SYNTH["target_genre"]]
    rows = []
    for g in g_names:
        rows.append({"genre": g, "cos_to_target": float(np.dot(tgt, centroids[g]))})
    tgt_df = pd.DataFrame(rows).sort_values("cos_to_target", ascending=False)
    print("\nTarget vector neighborhood for", SYNTH["target_genre"])
    display(tgt_df)


## 10) Source vs Generated Snippet Comparison

Direct A/B snippet comparison from the same timestamps in source and generated full-song outputs, with per-snippet metrics.

In [ ]:
SNIPPETS = {
    "starts_sec": [15, 45, 75, 105, 135],
    "duration_sec": 8.0,
    "sr": 22050,
    "max_to_render": 4,
}

print(SNIPPETS)


In [ ]:
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt

src_path = OUT / "source.wav"
gen_path = OUT / "longform_coherent.wav"
assert src_path.exists(), f"Missing {src_path}"
assert gen_path.exists(), f"Missing {gen_path}"

y_src, sr_src = librosa.load(src_path, sr=SNIPPETS["sr"], mono=True)
y_gen, sr_gen = librosa.load(gen_path, sr=SNIPPETS["sr"], mono=True)
assert sr_src == sr_gen == SNIPPETS["sr"]


def _clip(y, start_sec, dur_sec, sr):
    s = int(start_sec * sr)
    n = int(dur_sec * sr)
    e = min(len(y), s + n)
    clip = y[s:e]
    if len(clip) < n:
        clip = np.pad(clip, (0, n - len(clip)))
    return clip.astype(np.float32)


def _mel_db(y, sr):
    m = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=1024, hop_length=256, n_mels=80)
    return librosa.power_to_db(m, ref=np.max)


def _pitch_corr(a, b, sr):
    f0a, _, _ = librosa.pyin(a, fmin=50, fmax=2000, sr=sr, hop_length=256)
    f0b, _, _ = librosa.pyin(b, fmin=50, fmax=2000, sr=sr, hop_length=256)
    f0a = np.nan_to_num(f0a, nan=0.0)
    f0b = np.nan_to_num(f0b, nan=0.0)
    n = min(len(f0a), len(f0b))
    f0a, f0b = f0a[:n], f0b[:n]
    v = (f0a > 0) & (f0b > 0)
    if v.sum() < 10:
        return 0.0
    aa, bb = f0a[v], f0b[v]
    if aa.std() < 1e-6 or bb.std() < 1e-6:
        return 0.0
    c = float(np.corrcoef(aa, bb)[0, 1])
    return c if np.isfinite(c) else 0.0

rows = []
clips = []
for t0 in SNIPPETS["starts_sec"]:
    a = _clip(y_src, t0, SNIPPETS["duration_sec"], SNIPPETS["sr"])
    b = _clip(y_gen, t0, SNIPPETS["duration_sec"], SNIPPETS["sr"])

    ma = _mel_db(a, SNIPPETS["sr"])
    mb = _mel_db(b, SNIPPETS["sr"])

    mel_l1 = float(np.mean(np.abs(ma - mb)))
    mel_l2 = float(np.sqrt(np.mean((ma - mb) ** 2)))
    pcorr = _pitch_corr(a, b, SNIPPETS["sr"])
    rms_a = float(np.sqrt(np.mean(a ** 2)) + 1e-9)
    rms_b = float(np.sqrt(np.mean(b ** 2)) + 1e-9)

    rows.append({
        "start_sec": float(t0),
        "mel_l1_db": mel_l1,
        "mel_l2_db": mel_l2,
        "pitch_corr": pcorr,
        "rms_ratio_gen_to_src": rms_b / rms_a,
    })
    clips.append((t0, a, b, ma, mb))

snippet_df = pd.DataFrame(rows)
print("Per-snippet source vs generated metrics:")
display(snippet_df)

if (OUT / "coherence_metrics.json").exists():
    coh = json.loads((OUT / "coherence_metrics.json").read_text(encoding="utf-8"))
    print("\nRun-level coherence metrics:")
    display(pd.DataFrame([coh]))


In [ ]:
to_render = clips[: int(SNIPPETS["max_to_render"])]
for t0, a, b, ma, mb in to_render:
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    ax[0].imshow(ma, origin="lower", aspect="auto", cmap="magma")
    ax[0].set_title(f"Source @ {t0:.1f}s")
    ax[1].imshow(mb, origin="lower", aspect="auto", cmap="magma")
    ax[1].set_title(f"Generated @ {t0:.1f}s")
    ax[2].imshow(np.abs(ma - mb), origin="lower", aspect="auto", cmap="inferno")
    ax[2].set_title("|Mel diff|")
    plt.tight_layout()
    plt.show()

    print(f"Snippet @ {t0:.1f}s ? Source")
    display(ipd.Audio(a, rate=SNIPPETS["sr"]))
    print(f"Snippet @ {t0:.1f}s ? Generated")
    display(ipd.Audio(b, rate=SNIPPETS["sr"]))
